# Data Collection and Processing
Collects agent- and run-level data from all runs and validates them.

In [1]:
import json
import sys
from pathlib import Path
import pandas as pd
import polars as pl
import numpy as np

from exp_utils import (
    REPO_ROOT, graph_family, compute_agent_stats, directional_projection, leadlag_asymmetry, compute_run_stats, graph_bin,
    GRAPH_BIN_LABELS, STANCE_LABEL_TO_IDX,
)


# exp_utils already puts REPO_ROOT on sys.path.
sys.path.insert(0, str(REPO_ROOT / "notebooks"))
sys.path.insert(0, str(Path.cwd()))
from src.utils import move_invalid_runs


from src.analysis.loaders import load_adjacency_matrix, load_agents_data, load_runs_metadata
from src.analysis.loaders.loaders import load_network_data



In [2]:
SCENARIOS = ["base_llms", "random_roles", "random_experts", "experts"]

MATCHES = {
    ("llama-doc", "Clinical Physician"),
    ("llama-base", "LLM"),
    ("llama-base", "Human Participant"),
    ("llama-assistant", "Assistant"),
    ("llama-biomed", "Biomedical Researcher"),
    ("llama-chemist", "Chemist"),
    ("llama-coder", "Software Engineer"),
    ("llama-cyber", "Cybersecurity Analyst"),
    ("llama-finance", "Financial Analyst"),
    ("llama-hermes", "Strategic Planner"),
    ("llama-openmath", "Mathematician"),
    ("llama-lexicographer", "Lexicographer"),
    ("llama-linguist", "Language Mediator"),
    ("llama-roleplay", "Storyteller"),
    ("llama-scholar", "Academic Scholar"),
    ("llama-user", "Online Friend"),
}


def load_run_context(run_path):
    """config.json, the network manifest and agent ids: the only run files E1 needs besides beliefs."""
    run_path = Path(run_path)
    with open(run_path / "config.json") as f:
        config = json.load(f)
    with open(run_path / "results" / "agents_data.json") as f:
        agents_data = json.load(f)
    try:
        agent_ids = [v["node_id"] for v in agents_data.values()]
    except (KeyError, TypeError):
        agent_ids = [int(k) for k in agents_data]
    return config, load_network_data(run_path), agent_ids


def finalize_frame(df, cat_cols):
    """Shared dtype/ID post-processing for the agent- and run-level frames."""
    df["run_scenario"] = pd.Categorical(df["run_scenario"], categories=SCENARIOS)
    for c in cat_cols:
        df[c] = pd.Categorical(df[c], categories=sorted(df[c].unique()))
    df["graph_subtype"] = pd.Categorical(df["graph_degree_mean"].apply(graph_bin),
                                         categories=GRAPH_BIN_LABELS, ordered=True)
    run_id = df["run_scenario"].astype(str) + ":" + df["graph_id"].astype(str) + ":" + df["statement_id"].astype(str)
    df["run_id"] = pd.Categorical(run_id, categories=sorted(run_id.unique()))
    return df


## Load Data

In [3]:
# Load all runs
RUNS_DIR = REPO_ROOT / "data" / "outputs" / "runs"

df_total = pl.from_pandas(load_runs_metadata(RUNS_DIR))

# Filter for completed and validated runs
df_runs = df_total.filter(pl.col('validated') & pl.col('completed')).to_pandas()
df_runs["graph_family"] = df_runs["graph_type"].map(graph_family)

print(f"Run directories found : {len(df_total)}")
print(f"Total validated runs  : {len(df_runs)}")
print("\nRuns by setting:")
print(df_runs.groupby("setting").size())
print("\nRuns by graph family:")
print(df_runs.groupby("graph_family").size())
print(f"\nDistinct graph configs: {df_runs['graph_type'].nunique()}")

result_path = REPO_ROOT / "data" / "outputs" / "aggregated_expanded"
result_path.mkdir(parents=True, exist_ok=True)
print(f"\nWriting aggregates to: {result_path}")

Run directories found : 2560
Total validated runs  : 2560

Runs by setting:
setting
base_llms         640
experts           640
random_experts    640
random_roles      640
dtype: int64

Runs by graph family:
graph_family
erdos-renyi       1280
watts-strogatz    1280
dtype: int64

Distinct graph configs: 18

Writing aggregates to: <repo>/data/outputs/aggregated_expanded


In [4]:
print("Check run distribution across settings and statements:")
df_runs.groupby(["setting","statement_id"]).agg("size").unstack(fill_value=0)

Check run distribution across settings and statements:


statement_id,false_0,false_1,false_2,false_3,false_4,false_5,false_6,false_7,false_8,false_9,true_0,true_1,true_2,true_3,true_4,true_5,true_6,true_7,true_8,true_9
setting,,,,,,,,,,,,,,,,,,,,
base_llms,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32
experts,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32
random_experts,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32
random_roles,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32,32


In [5]:
# Dry run first: see exactly what would be moved before moving anything.
#
# `load_runs_metadata` records a run directory as unvalidated when it has no
# config.json (it cannot be attributed to a seed), so config-less-but-finished
# directories are swept here too rather than lingering unseen.
invalid = df_total.filter(~(pl.col("validated") & pl.col("completed"))).to_pandas()
print(f"Invalid run directories: {len(invalid)}")
if len(invalid):
    display(
        invalid[["setting", "graph_type", "statement_id", "datetime",
                 "seed", "completed", "validated", "run_path"]]
    )

stats = move_invalid_runs(
    runs_dir=RUNS_DIR,
    incomplete_dir=REPO_ROOT / "data" / "outputs" / "incomplete_runs",
    dry_run=True,
)
print(stats)

Invalid run directories: 0


{'incomplete': 0, 'unvalidated': 0, 'total': 0}


## 1. Processing (Agent-based)

In [6]:
lf = (
    pl.from_pandas(df_runs)
    .filter(pl.col("setting").is_in(SCENARIOS))
    .select(["run_path", "setting", "seed", "graph_type", "statement_id", 'graph_family'])
    .with_columns(graph_id=pl.format("{}-{}", "graph_type", "seed"))
)

lf.head()

run_path,setting,seed,graph_type,statement_id,graph_family,graph_id
str,str,i64,str,str,str,str
"""/…","""base_llms""",814183,"""watts-strogatz""","""false_0""","""watts-strogatz""","""watts-strogatz-814183"""
"""/…","""base_llms""",1170252924,"""watts-strogatz""","""false_0""","""watts-strogatz""","""watts-strogatz-1170252924"""
"""/…","""base_llms""",900911955,"""watts-strogatz""","""false_0""","""watts-strogatz""","""watts-strogatz-900911955"""
"""/…","""base_llms""",473392625,"""watts-strogatz""","""false_0""","""watts-strogatz""","""watts-strogatz-473392625"""
"""/…","""base_llms""",964669078,"""watts-strogatz""","""false_0""","""watts-strogatz""","""watts-strogatz-964669078"""


In [7]:
recs = []

for run in lf.iter_rows(named=True):
    
    run_agents = load_agents_data(run["run_path"])
    config, manifest, agent_ids = load_run_context(run["run_path"])
    A, _ = load_adjacency_matrix(run["run_path"])

    belief_ful = run_agents["belief_ful"]  # (N_agents, N_rounds, 3)
    _models = run_agents["models"]
    _roles = run_agents["roles"]
    _network_features = run_agents["network_features"]
    ground_truth_class_idx = STANCE_LABEL_TO_IDX[config['statement']['stats']['label_ground_truth']]

    graph_id = run["graph_id"]
    graph_seed = str(config['network']['seed_map'][str(run["seed"])])
    g = manifest['graph']

    # Neighbours vs non-neighbours of the same run: non-neighbours cannot see i at lag 1.
    influence = directional_projection(belief_ful, A)
    # Who moves first relative to whom: "i then j" minus "j then i" on each edge.
    leadlag = leadlag_asymmetry(belief_ful, A)

    for i in range(belief_ful.shape[0]):
        stats = compute_agent_stats(belief_ful[i], ground_truth_class_idx)  # (N_rounds, 3) per agent
        model, role, nf = _models[i], _roles[i], _network_features[i]
        recs.append(
            {
                # agent info
                "agent_id": agent_ids[i],
                "agent_model": model,
                "agent_role": role,
                "agent_is_matched": (model, role) in MATCHES,
                "agent_is_expert": not model.startswith("llama-base"),
                # node info
                "node_degree": float(nf["degree"]),
                "node_centrality_eigen": float(nf["eigenvector_centrality"]),
                "node_centrality_between": float(nf["betweenness_centrality"]),
                "node_clustering": float(nf["clustering_coefficient"]),
                # graph info
                "graph_id": graph_id,
                "graph_seed": graph_seed,
                "graph_family": run["graph_family"],
                "graph_degree_mean": float(g['degrees']['mean']),
                "graph_degree_median": float(g['degrees']['median']),
                "graph_sp": float(g["avg_path_length"]),
                # run info
                "run_seed": run["seed"],
                "run_scenario": run["setting"],
                "run_T": stats["T"],
                "statement_id": run['statement_id'],
                # agent dynamics 
                "agent_plasticity": stats["plasticity_tv"],
                "agent_net_belief_change": stats["net_belief_change"], # net distance between final and initial P-distribution
                "agent_net_correct_shift": stats["net_correct_shift"],
                "agent_progress_ratio": stats["progress_ratio"],
                "agent_progress_forward": stats["progress_forward"],
                "agent_progress_backward": stats["progress_backward"],
                "agent_progress_efficiency": stats["progress_efficiency"],
                "agent_monotonicity": stats["monotonicity"],
                "agent_accuracy": stats["agent_accuracy"],
                # influence: neighbours' next move along i's direction, per unit of i's move
                "agent_influence_proj_out": influence["proj_out"][i],
                "agent_influence_proj_non": influence["proj_non"][i],
                "agent_influence_proj_contrast": influence["proj_contrast"][i],
                # lead-lag asymmetry: net_cos in [-1, 1] (primary), net_dot in L2^2 per round
                "agent_leadlag_net_cos": leadlag["net_cos"][i],
                "agent_leadlag_net_dot": leadlag["net_dot"][i],
            }
        )
dft = pd.DataFrame(recs)

for c in ["node_degree", "node_centrality_eigen", "node_centrality_between", "node_clustering"]:
    dft[f"{c}_z"] = (dft[c] - dft[c].mean()) / dft[c].std()

dft = finalize_frame(
    dft,
    cat_cols=["agent_model", "graph_family", "agent_role", "statement_id", "graph_seed"],
)

dft.to_parquet(result_path / "agent_level_data.parquet", index=False)


## Processing (Run-based)

In [8]:
recs = []

for run in lf.iter_rows(named=True):
    run_agents = load_agents_data(run["run_path"])
    config, net_data, _ = load_run_context(run["run_path"])

    belief_ful = run_agents["belief_ful"]  # (N_agents, N_rounds, 3)
    belief_cat = run_agents["belief_cat"]  # (N_agents, N_rounds); categorical belief per agent per round
    ground_truth = config['statement']['stats']['label_ground_truth']
    ground_truth_class_idx = STANCE_LABEL_TO_IDX[ground_truth]

    run_stats = compute_run_stats(belief_ful, ground_truth_class_idx)

    # Convergence: no agent flips its categorical belief state on the last round
    # (i.e. its stated belief at round T equals its belief at round T-1).
    flipped_last_round = belief_cat[:, -1] != belief_cat[:, -2]
    n_agents_flipped_last_round = int(flipped_last_round.sum())
    converged = n_agents_flipped_last_round == 0

    recs.append(
        {
                # run info
                "run_T": run_stats["T"],
                "run_scenario": run["setting"],
                "run_seed": run["seed"],
                "run_n_agents": belief_cat.shape[0],
                "statement_id": run['statement_id'],
                "statement_ground_truth": ground_truth,
                # graph info
                "graph_id": run["graph_id"],
                "graph_seed": str(config['network']['seed_map'][str(run["seed"])]),
                "graph_family": run["graph_family"],
                "graph_degree_mean": float(net_data['graph']['degrees']['mean']),
                "graph_degree_median": float(net_data['graph']['degrees']['median']),
                "graph_degree_std": net_data['graph']['degrees']['std'],
                "graph_sp": float(net_data['graph']["avg_path_length"]),
                "graph_num_nodes": net_data['graph']['num_nodes'],
                "graph_num_edges": net_data['graph']['num_edges'],
                "graph_radius": net_data['graph']['radius'],
                "graph_diameter": net_data['graph']['diameter'],
                "graph_triangles": net_data['graph']['triangles'],
                "graph_wedges": net_data['graph']['wedges'],
                "graph_num_components": net_data['graph']['num_components'],
                "graph_avg_clustering": net_data['graph']['avg_clustering'],
                "graph_global_clustering": net_data['graph']['global_clustering_coefficient'],
                "graph_density": net_data['graph']['density'],
                # population statistics
                "pop_agents_flipped_T": n_agents_flipped_last_round,
                "pop_agents_converged": converged,
                
                # consensus
                "pop_consensus_T": run_stats["modal_consensus_T"],
                "pop_consensus_0": run_stats["modal_consensus_0"],
                "pop_consensus_mean": run_stats["modal_consensus_mean"],
                "pop_consensus_change": run_stats["modal_consensus_T"] - run_stats["modal_consensus_0"],
                # last round statistics
                "pop_num_distinct_labels_T": run_stats["n_distinct_labels_T"],
                "pop_majority_label_T": run_stats["majority_label_T"],
                "pop_majority_share_T": run_stats["majority_share_T"],
                #accuracy
                "pop_accuracy_0": run_stats["accuracy_0"],
                "pop_accuracy_T": run_stats["accuracy_T"],
                "pop_accuracy_diff": run_stats["diff_accuracy"],
                # population polarization
                "pop_polarization_T": run_stats["polarization_T"],
                "pop_polarization_0": run_stats["polarization_0"],
                "pop_polarization_mean": run_stats["polarization_mean"],
                "pop_polarization_change": run_stats["polarization_T"] - run_stats["polarization_0"],
                'pop_polarization_change_1': run_stats["polarization_T"] - run_stats["polarization_1"],
                # entropy
                "pop_entropy_T": run_stats["entropy_T"],
                "pop_entropy_0": run_stats["entropy_0"],
                "pop_entropy_change": run_stats["entropy_change"],
        }
    )
dft = finalize_frame(
    pd.DataFrame(recs),
    cat_cols=["graph_family", "statement_id", "graph_seed"],
)

dft.to_parquet(result_path / "run_level_data.parquet", index=False)
